# PCNN -- Beispiel 2: MeteoSchweiz-Wetterdaten (Station wählbar)

Santiago Rojo Osorio

Dieses Notebook nutzt **echte Gebäudedaten** (`Data_processed.csv`) zusammen mit
**echten Stationsmessungen** von MeteoSchweiz (SwissMetNet).

Die Wetterstation kann auf zwei Arten gewählt werden (`STATION_MODE` unten):

- `"abbr"` -- Stationskürzel direkt angeben (z. B. `"STG"` für St. Gallen).
- `"coords"` -- nur Koordinaten (lat/lon) angeben; das Skript sucht automatisch
  die **nächstgelegene** MeteoSchweiz-Station (Haversine-Distanz über
  `weather_sources.nearest_station()`).

Funktioniert nur für Standorte **innerhalb der Schweiz** (SwissMetNet-Netz).
Für Standorte außerhalb der Schweiz siehe `example_03_openmeteo.ipynb` (Open-Meteo,
weltweit, nur Koordinaten).

**Hinweis:** Dieses Notebook lädt Daten live aus dem Internet
(`data.geo.admin.ch`) -- es muss lokal mit Internetzugang ausgeführt werden.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pcnn_model import PCNNModel, print_pcnn_result, print_energy_balance
from weather_sources import list_meteoswiss_stations, nearest_station, fetch_meteoswiss_hourly

## 1. Konfiguration

Hier alles anpassen, was für dein Gebäude/deinen Datensatz gilt.

In [ ]:
# ─── Gebäudedaten ────────────────────────────────────────────────────────────
DATA_PATH = "data/heizdaten_beispiel.csv"          # Pfad zur aufbereiteten Messreihe
QCOL      = "Waermeleistung_gemessen"        # oder "Waermeleistung" (Modellwert statt Messung)

# ─── Wetterstation wählen ────────────────────────────────────────────────────
STATION_MODE  = "coords"   # "abbr" -> STATION_ABBR direkt nutzen
                            # "coords" -> nächstgelegene Station zu (LAT, LON) suchen

STATION_ABBR  = "STG"      # nur relevant wenn STATION_MODE == "abbr"
LAT, LON      = 47.4239, 9.3767   # nur relevant wenn STATION_MODE == "coords" (Beispiel: St. Gallen)

# ─── Zeitraum ────────────────────────────────────────────────────────────────
# An den tatsächlich in Data_processed.csv vorhandenen Zeitraum anpassen
# (siehe Ausgabe von Zelle weiter unten: df_real.index.min() / .max()).
START = "2026-03-01"
END   = "2026-03-31 23:00"

## 2. Wetterstation bestimmen

Bei `STATION_MODE = "coords"` zeigt die Tabelle die 5 nächstgelegenen Stationen -
so lässt sich prüfen, ob die automatische Wahl plausibel ist, bevor die Daten
heruntergeladen werden.

In [ ]:
if STATION_MODE == "coords":
    stations = list_meteoswiss_stations()
    kandidaten = nearest_station(LAT, LON, stations=stations, n=5)
    display(kandidaten)
    station_abbr = kandidaten.iloc[0]["station_abbr"]
    print(f"\n-> gewählt: {station_abbr}  ({kandidaten.iloc[0]['station_name']}, "
          f"{kandidaten.iloc[0]['distanz_km']:.1f} km entfernt)")
elif STATION_MODE == "abbr":
    station_abbr = STATION_ABBR
    print(f"-> Station direkt gewählt: {station_abbr}")
else:
    raise ValueError("STATION_MODE muss 'abbr' oder 'coords' sein.")

## 3. Gebäudedaten laden (`Data_processed.csv`)

In [ ]:
df_real = pd.read_csv(
    DATA_PATH,
    usecols=["timestamp", "Waermeleistung", "Waermeleistung_gemessen"],
    parse_dates=["timestamp"],
    index_col="timestamp",
)
print(f"Datenbereich verfügbar: {df_real.index.min()}  bis  {df_real.index.max()}")

df_real_1h = df_real.resample("1h").mean()
df_real_1h = df_real_1h.loc[START:END]
print(f"Verwendeter Zeitraum : {df_real_1h.index.min()}  bis  {df_real_1h.index.max()}  "
      f"({len(df_real_1h)} Stunden)")

## 4. Wetterdaten der gewählten Station laden

In [ ]:
weather = fetch_meteoswiss_hourly(station_abbr, START, END)
weather.head()

## 5. Gebäude- und Wetterdaten zusammenführen

In [ ]:
df_join = df_real_1h.join(weather, how="inner").dropna()
print(f"Gemeinsamer Zeitraum: {len(df_join)} Stunden")
df_join[[QCOL, "T_a", "W_s", "I_g"]].head()

## 6. PCNN trainieren

In [ ]:
model = PCNNModel(epochs=1000, verbose=True)
result = model.fit(
    df_join[QCOL].values,
    df_join["T_a"].values,
    df_join["W_s"].values,
    df_join["I_g"].values,
)
print_pcnn_result(result)

## 7. Vorhersage vs. Messung (Validierungsperiode)

In [ ]:
n_val = result.n_val
t_val = np.arange(n_val)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(t_val, result.Q_true, label=f"{QCOL} (real)", color="#185FA5")
ax.plot(t_val, result.Q_pred, label="PCNN Q_hat", color="#E24B4A", ls="--")
ax.set_xlabel("Stunde (Validierungsperiode)"); ax.set_ylabel("Q_h [W]")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()

## 8. Energieverteilung

In [ ]:
T_a_val = df_join["T_a"].values[-n_val:]
W_s_val = df_join["W_s"].values[-n_val:]
I_g_val = df_join["I_g"].values[-n_val:]

energie = model.energy_balance(T_a_val, W_s_val, I_g_val)
print_energy_balance(energie, label=f"{station_abbr} -- {START} bis {END}")